In [1]:
!pip install --upgrade langchain-openai
!pip install --upgrade langchain
!pip install --upgrade langchain-core
!pip install --upgrade langchain_huggingface


!pip install qdrant-client


[notice] A new release of pip available: 22.3 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip available: 22.3 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip available: 22.3 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip available: 22.3 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip available: 22.3 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http import models
import re
import typing as tp
from nltk.tokenize import word_tokenize
# from rank_bm25 import BM25Okapi
# from langchain_core.documents import Document
# from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain_huggingface import HuggingFaceEmbeddings
# from langchain.vectorstores import Qdrant as LCQdrant
from tqdm import tqdm


c:\Users\cody\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# COLLECTION_NAME = "freecad_docs"
# DATA_PATH = "data"
# DOCS_PATH = os.path.join(DATA_PATH, "docs")
# EMBEDDINGS_PATH = "embeddings"
# QDRANT_PATH = "qdrant"

COLLECTION_NAME = "freecad_docs"
DATA_PATH = "/kaggle/input"
DOCS_PATH = os.path.join(DATA_PATH, "docs")
EMBEDDINGS_PATH = "embeddings"
QDRANT_PATH = "/kaggle/working/freecad-qdrant"

In [4]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

# Metrics

In [5]:
def is_relevant(doc_sample, ground_truth):
    pass

In [6]:
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

In [7]:
def mean_average_cosine_similarity(similarities: tp.List[tp.List[float]]) -> float:
    return sum(sum(sims) / len(sims) for sims in similarities) / len(similarities)

In [8]:
def mean_max_cosine_similarity(similarities: tp.List[tp.List[float]]) -> float:
    return sum(max(sims) for sims in similarities) / len(similarities)

In [9]:
def hit_rate_at_k(similarities: tp.List[tp.List[float]], threshold: float, k: int) -> float:
    hits = 0
    for sims in similarities:
        top_k = sims[:k]
        if any(sim >= threshold for sim in top_k):
            hits += 1
    return hits / len(similarities)

# Qdrant init

In [10]:
!cp -r /kaggle/input/freecad-qdrant/qdrant /kaggle/working/freecad-qdrant

"cp" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


In [11]:
embedding_model = SentenceTransformer("BAAI/bge-m3", device=device)

In [12]:
client = QdrantClient(path=QDRANT_PATH)

In [13]:
def filter_code(doc: str) -> str:
    code_blocks = re.findall(r"```(?:[a-zA-Z]+\n)?(.*?)```", doc, re.DOTALL)
    code = "\n\n".join(code_blocks)
    return re.sub(r'\n{3,}', '\n\n', code)

In [14]:
query_df = pd.read_csv(os.path.join(DATA_PATH, "freecad-bench/tooltips_dataset.csv"))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input\\freecad-bench/tooltips_dataset.csv'

# Qdrant LC init

In [18]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": device})
vectordb = LCQdrant.from_existing_collection(
    path="qdrant",
    embedding=embeddings,
    collection_name="freecad_docs_lc",
)

NameError: name 'LCQdrant' is not defined

In [ ]:
vectordb.similarity_search("Window")

[Document(metadata={'source': 'Reinforcement_ColumnRebars.wikitext', 'chunk_idx': 6, '_id': 'e6546e97-2147-4dab-a27d-1e162f8902ec', '_collection_name': 'freecad_docs_lc'}, page_content='Example'),
 Document(metadata={'source': 'Arch_Site.wikitext', 'chunk_idx': 2, '_id': '703319b2-08ac-44d3-aa6c-9b12a0b95317', '_collection_name': 'freecad_docs_lc'}, page_content='Properties\n\n Data'),
 Document(metadata={'source': 'Arch_Window.wikitext', 'chunk_idx': 10, '_id': 'a2369c9d-973b-44c0-9821-69aeff535f1c', '_collection_name': 'freecad_docs_lc'}, page_content='Our window type is now ready. We can create window objects from it, simply by selecting the App Part and pressing the window button. The "Height", "Width", "Subvolume" and "Tag" properties of the window will be linked to the corresponding property of the App Part, if existing.\n\nMaterials\n\nTo build a material for type-based windows:\n Create a multi-material\n Create one entry in the multi-material for each component of your App Par

# BM25 Baseline

In [15]:
for doc in corpus:
    print(hf_tokenizer.tokenize(doc))
    break

NameError: name 'corpus' is not defined

In [ ]:
query_idx = 2

In [19]:
docs_df = pd.read_csv(os.path.join(DATA_PATH, "chunks_df.csv"))
corpus = docs_df["content"].values

tokenized_corpus = []
for doc in corpus:
    # tokenized_corpus.append(word_tokenize(filter_code(str(doc))))
    tokenized_corpus.append(word_tokenize(str(doc)))


bm25 = BM25Okapi(tokenized_corpus)

def bm25_retrieve(query: str, k: int = 5):
    tokenized_query = query.split()
    scores = bm25.get_scores(tokenized_query)
    top_k_idx = np.argsort(scores)[::-1][:k]
    return [corpus[i] for i in top_k_idx], [scores[i] for i in top_k_idx]

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input\\chunks_df.csv'

In [16]:
def bm25_retrieve_with_score(queries: np.ndarray, y: np.ndarray, k: int = 5):
    retrieved_texts = []
    scores = []

    for i in range(len(queries)):
        results, _ = bm25_retrieve(queries[i], k=k)
        retrieved_texts.append(results)

        retrieved_vecs = embedding_model.encode([filter_code(text) for text in results])
        y_vec = embedding_model.encode(y[i])

        scores_batch = [cosine_similarity(vec, y_vec) for vec in retrieved_vecs]
        scores.append(scores_batch)

    return retrieved_texts, scores

In [17]:
query = query_df.iloc[query_idx]["X"]
similarities, scores = bm25_retrieve(query)
print(query)
for i, (text, score) in enumerate(zip(similarities, scores)):
    print(f"=================Hit {i+1} score: {score} =================")

    print(f"{text}\n")

NameError: name 'query_df' is not defined

# Experiments

In [20]:
import logging
import numpy as np
from tqdm import tqdm
from rank_bm25 import BM25Okapi

logger = logging.getLogger("bm25_grid_search")
logger.setLevel(logging.INFO)

file_handler = logging.FileHandler("bm25_grid_search.log", mode='w')
file_handler.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter('%(message)s'))

if not logger.hasHandlers():
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

def bm25_grid_search(documents, query_df, embedder, top_k=10):
    k1_values = [0.5, 1.0, 1.5, 2.0]
    b_values = [0.0, 0.25, 0.5, 0.75, 1.0]

    tokenized_corpus = [doc.split() for doc in documents]
    results = []

    logger.info("BM25 Grid Search Started...")
    logger.info(f"Corpus size: {len(documents)} documents")
    logger.info(f"Queries: {len(query_df)}")
    logger.info(f"Top K: {top_k}")
    logger.info(f"Embedder: {embedder.__class__.__name__}")

    for k1 in k1_values:
        for b in b_values:
            logger.info(f"\n--- Testing params: k1={k1}, b={b} ---")
            bm25 = BM25Okapi(tokenized_corpus, k1=k1, b=b)

            retrieved_docs_list = []
            for query in tqdm(query_df["X"], desc=f"k1={k1}, b={b}"):
                scores = bm25.get_scores(query.split())
                top_k_idx = np.argsort(scores)[::-1][:top_k]
                retrieved = [documents[i] for i in top_k_idx]
                retrieved_docs_list.append(retrieved)

            macs = mean_average_cosine_similarity(retrieved_docs_list, query_df["Y"].tolist(), embedder)
            mmcs = mean_max_cosine_similarity(retrieved_docs_list, query_df["Y"].tolist(), embedder)
            hit = hit_rate_at_k(retrieved_docs_list, query_df["Y"].tolist(), embedder, k=top_k)

            logger.info(f"MACS:  {macs:.4f}")
            logger.info(f"MMCS:  {mmcs:.4f}")
            logger.info(f"Hit@{top_k}: {hit:.4f}")

            results.append({
                "k1": k1,
                "b": b,
                "MACS": macs,
                "MMCS": mmcs,
                "Hit@K": hit
            })

    best = max(results, key=lambda x: x["MACS"])
    logger.info(f"Best config: k1={best['k1']}, b={best['b']}, MACS={best['MACS']:.4f}")
    return sorted(results, key=lambda x: x["MACS"], reverse=True)


In [21]:
query_idx = 0

query = query_df.iloc[query_idx]["X"]
query_qdrant = query + " python code"
# query_qdrant = "Add window"
# query_qdrant = "Make a circle"


query_vec = embedding_model.encode(query_qdrant)
hits = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vec,
    limit=5,
)

for _ in range(len(hits.points)):
    print(hits.points[_].payload['source'])

retrieved_text_batch = [point.payload['content'] for point in hits.points]
retrieved_code_batch = [filter_code(text) for text in retrieved_text_batch]

retrieved_vec = embedding_model.encode(retrieved_code_batch)

y_vec = embedding_model.encode(query_df.iloc[query_idx]["Y"])

print(f"X: {query_df.iloc[query_idx]['X']}")
print("Y\n", query_df.iloc[query_idx]["Y"], "\n")
for i, vec in enumerate(retrieved_vec):
    print(f"Hit {i+1}, score: {cosine_similarity(vec, y_vec)}")

for i, text in enumerate(retrieved_text_batch):
    print(f"=================Hit {i+1}=================")
    print(text)


NameError: name 'query_df' is not defined

In [22]:
def retrieve_with_score(queries: np.ndarray, y: np.ndarray, k: int):
    for i in range(len(queries)):
        queries[i] += " python code"

    query_vectors = embedding_model.encode(queries)

    retrieved_texts = []
    scores = []

    for i in range(len(queries)):
        hits = client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vectors[i],
            limit=k,
        )

        retrieved_text_batch = [point.payload['content'] for point in hits.points]
        retrieved_code_batch = [filter_code(text) for text in retrieved_text_batch]


        retrieved_vec = embedding_model.encode(retrieved_code_batch)
        y_vec = embedding_model.encode(y[i])

        scores_batch = [cosine_similarity(x_vec, y_vec) for x_vec in retrieved_vec]

        scores.append(scores_batch)
        retrieved_texts.append(retrieved_text_batch)

    return retrieved_texts, scores

In [23]:
queries = query_df["X"].values
y = query_df["Y"].values
texts, scores = retrieve_with_score(
    queries=queries,
    y=y,
    k=5
)

NameError: name 'query_df' is not defined

In [ ]:
# import json

# retrieved_df = query_df.copy()
# retrieved_df["retrieved"] = texts
# retrieved_df["retrieved"] = retrieved_df["retrieved"].apply(json.dumps)
# retrieved_df.to_csv("bench_retrieved.csv", index=False)

In [24]:
bm_texts, bm_scores = bm25_retrieve_with_score(
    queries=queries,
    y=y,
    k=5
)

NameError: name 'queries' is not defined

In [ ]:
# Qdrant
# MACS 0.5307003046751022
# MMCS 0.6132178441286087
# Hit rate cosine 0.604

print("Qdrant")
print("MACS", mean_average_cosine_similarity(scores))
print("MMCS", mean_max_cosine_similarity(scores))
print("Hit rate cosine", hit_rate_at_k(scores, 0.6, 5))

Qdrant
MACS 0.5281056774377827
MMCS 0.6029993942975997
Hit rate cosine 0.512


In [ ]:
print("BM25")
print("MACS", mean_average_cosine_similarity(bm_scores))
print("MMCS", mean_max_cosine_similarity(bm_scores))

BM25
MACS 0.46777109932899463
MMCS 0.5069063054323196


In [ ]:
def rrf_fusion(bm25_docs, qdrant_docs, k_rrf=60, top_k=5):
    fused_scores = {}
    for rank, doc in enumerate(bm25_docs):
        fused_scores[doc] = fused_scores.get(doc, 0) + 1 / (k_rrf + rank + 1)
    for rank, doc in enumerate(qdrant_docs):
        fused_scores[doc] = fused_scores.get(doc, 0) + 1 / (k_rrf + rank + 1)
    sorted_docs = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return [doc for doc, score in sorted_docs[:top_k]]


In [ ]:
def rrf_retrieve_with_score(queries: np.ndarray, y: np.ndarray, client, bm25_obj, embedding_model, collection_name, k=5, k_rrf=60):
    all_texts = []
    all_scores = []

    for i in tqdm(range(len(queries))):
        query = queries[i]

        bm25_docs, _ = bm25_retrieve(query, k=k)

        query_vec = embedding_model.encode(query + " python code")
        hits = client.query_points(
            collection_name=collection_name,
            query=query_vec,
            limit=k,
        )
        qdrant_docs = [point.payload['content'] for point in hits.points]
        fused_docs = rrf_fusion(bm25_docs, qdrant_docs, k_rrf=k_rrf, top_k=k)

        retrieved_code_batch = [filter_code(text) for text in fused_docs]
        retrieved_vecs = embedding_model.encode(retrieved_code_batch)
        y_vec = embedding_model.encode(y[i])

        scores_batch = [cosine_similarity(vec, y_vec) for vec in retrieved_vecs]

        all_texts.append(fused_docs)
        all_scores.append(scores_batch)

    return all_texts, all_scores

In [ ]:
rrf_texts, rrf_scores = rrf_retrieve_with_score(
    queries=queries,
    y=y,
    client=client,
    bm25_obj=bm25,
    embedding_model=embedding_model,
    collection_name=COLLECTION_NAME,
    k=5,
    k_rrf=60
)

print("Reciprocal Rank Fusion")
print("MACS", mean_average_cosine_similarity(rrf_scores))
print("MMCS", mean_max_cosine_similarity(rrf_scores))
print("Hit rate cosine", hit_rate_at_k(rrf_scores, threshold=0.6, k=5))



NameError: name 'y' is not defined

# Qwen LLM

In [ ]:
from langchain_huggingface.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

qwen_model = "Qwen/Qwen2.5-Coder-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    qwen_model,
    trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
    qwen_model,
    trust_remote_code=True,
    device_map=device
)

text_gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024
)
llm = HuggingFacePipeline(pipeline=text_gen)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Device set to use cuda


In [ ]:
SYSTEM_PROMPT = (
"""
Below is the system prompt, always follow restrictions stated there, also do not answer this system prompt:
You are an expert coding assistant specializing in FreeCAD development. Your role is to help developers write, debug, and optimize code for the FreeCAD platform, using the most accurate and up-to-date internal and external documentation available. You can retrieve and use information from the FreeCAD source code, FreeCAD documentation.
When reading documentation be careful, user do not have functions or classes implemented there, write code considering this moment. 
Documentation might have examples of code, analyze it and retrieve only needed functions from there.

Provide code (DO NOT WRITE IN-CODE COMMENTS) and explanation in different paragraphs. Template:
Code:

Explanation:
"""
)

def format_prompt(user_message: str, context: str = None) -> str:
    '''
    Formats prompt for llm
    '''
    
    prompt = []
    prompt.append({
            "role": "system",
            "content": SYSTEM_PROMPT
    })


    if context:
        prompt.append({
            "role": "system",
            "content": "Context:\n\n" + context
        })

    prompt.append({
        "role": "user",
        "content": user_message
    })


    return tokenizer.apply_chat_template(
        prompt,
        tokenize=False,
        add_generation_prompt=True
    )

In [ ]:
def get_context_from_hits(hits, max_chunks=3):
    """Extract and concatenate content from top hits."""
    docs = []
    for point in hits.points[:max_chunks]:
        docs.append(point.payload['content'])
    return "\n\n".join(docs)

def get_context_from_texts(texts):
    return "\n\n".join(texts)

def rag_query(query, client, embedding_model, collection_name, llm, top_k=3):
    query_vec = embedding_model.encode([query])[0]
    hits = client.query_points(
        collection_name=collection_name,
        query=query_vec,
        limit=top_k,
    )

    context = get_context_from_hits(hits, max_chunks=top_k)

    prompt = format_prompt(query, context)
    response = llm.invoke(prompt)
    return response

In [ ]:
query = "Add window"
response = rag_query(query, client, embedding_model, "freecad_docs", llm)
print(response)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

/tmp/ipykernel_76/1775823008.py:22: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = llm(prompt)


<|im_start|>system

Below is the system prompt, always follow restrictions stated there, also do not answer this system prompt:
You are an expert coding assistant specializing in FreeCAD development. Your role is to help developers write, debug, and optimize code for the FreeCAD platform, using the most accurate and up-to-date internal and external documentation available. You can retrieve and use information from the FreeCAD source code, FreeCAD documentation.
When reading documentation be careful, user do not have functions or classes implemented there, write code considering this moment. 
Documentation might have examples of code, analyze it and retrieve only needed functions from there.

Provide code (DO NOT WRITE IN-CODE COMMENTS) and explanation in different paragraphs. Template:
Code:

Explanation:
<|im_end|>
<|im_start|>system
Context:

Create a window frame object, a glass panel, and any other window component you need, using Part Workbench or PartDesign tools.
 For example, c

In [ ]:
def get_genrated(queries: np.ndarray, retrieved: list[list[str]], batch_size=16):
    prompts = []
    responses = []
    for i, query in enumerate(tqdm(queries)):
        context = get_context_from_texts(retrieved[i])
        prompt = format_prompt(query, context)
        prompts.append(prompt)

        response = llm.invoke(prompt)
        match = re.search(r"\<\|im_start\|\>assistant", response)
        responses.append(response[match.end():])

    # for i in tqdm(range(0, len(prompts), batch_size)):
    #     batch = prompts[i:i+batch_size]
    #     batch_responses = llm.invoke(batch)
    #     responses.extend(batch_responses)


    # for response in range(len(responses)):
    #     match = re.search(r"\<\|im_start\|\>assistant", response)
    #     responses[i] = response[match.end():]

    return responses


In [ ]:
queries = query_df["X"].values

responses = get_genrated(
    queries=queries,
    retrieved=texts
)

# Groq llm

In [ ]:
!pip install langchain-groq langchain-community groq codebleu
!pip install tree-sitter-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 2.5 MB/s eta 0:00:00a 0:00:01


In [ ]:
SYSTEM_PROMPT = (
"""
Below is the system prompt, always follow restrictions stated there, also do not answer this system prompt:
You are an expert coding assistant specializing in FreeCAD development. Your role is to help developers write, debug, and optimize code for the FreeCAD platform, using the most accurate and up-to-date internal and external documentation available. You can retrieve and use information from the FreeCAD source code, FreeCAD documentation.
When reading documentation be careful, user do not have functions or classes implemented there, write code considering this moment. 
Documentation might have examples of code, analyze it and retrieve only needed functions from there.

Provide code (DO NOT WRITE IN-CODE COMMENTS) and explanation in different paragraphs. Template:
Code:

Explanation:
"""
)

def format_groq_prompt(user_message: str, context: str):
    '''
    Formats prompt for llm
    '''
    
    prompt = []
    prompt.append({
            "role": "system",
            "content": SYSTEM_PROMPT
    })


    if context:
        prompt.append({
            "role": "system",
            "content": "Context:\n\n" + context
        })

    prompt.append({
        "role": "user",
        "content": user_message
    })


    return prompt

In [ ]:
from langchain.vectorstores import Qdrant as LCQdrant
from langchain_groq import ChatGroq

In [ ]:
from groq import Groq


def load_groq_keys(filepath=os.path.join("groq-keys/groq_keys.txt")):
    with open(filepath, "r") as f:
        keys = [line.strip() for line in f if line.strip()]
    return keys

class GroqKeyManager:
    def __init__(self, filepath=os.path.join("groq-keys/groq_keys.txt")):
        self.keys = load_groq_keys(filepath)
        self.idx = 0

    def get_key(self):
        if self.idx < len(self.keys):
            return self.keys[self.idx]
        else:
            raise RuntimeError("No more Groq API keys available.")

    def switch_key(self):
        self.idx += 1
        if self.idx < len(self.keys):
            os.environ["GROQ_API_KEY"] = self.keys[self.idx]
            return self.keys[self.idx]
        else:
            raise RuntimeError("No more Groq API keys available.")


groq_key_manager = GroqKeyManager(os.path.join(DATA_PATH, "groq-keys/groq_keys.txt"))
os.environ["GROQ_API_KEY"] = groq_key_manager.get_key()


def get_groq_client():
    return Groq(api_key=os.environ["GROQ_API_KEY"])


In [ ]:
def generate_groq(prompt, model_name):
    client = get_groq_client()

    chat_completion = client.chat.completions.create(
        messages=prompt,
        model=model_name,
    )

    return chat_completion.choices[0].message.content

In [ ]:
from time import sleep

responses = []
last_key_idx = 0

def get_groq_genrated(
        model_name: str,
        queries: np.ndarray,
        retrieved: list[list[str]]
    ):
    prompts = []
    global client


    for i, query in enumerate(tqdm(queries)):
        context = get_context_from_texts(retrieved[i])
        prompt = format_groq_prompt(query, context)
        prompts.append(prompt)

        error_patience = 0
        while True:
            try:
                responses.append(generate_groq(prompt, model_name))
                break
            except Exception as e:
                if "rate limit" in str(e).lower() or "TPD" in str(e).lower():
                    try:
                        groq_key_manager.switch_key()
                        client = get_groq_client()
                        print(f"Switched to next GROQ API key: {groq_key_manager.idx}")
                    except RuntimeError:
                        print("All GROQ API keys exhausted.")
                        responses.append(None)
                        break
                else:
                    error_patience += 1
                    print(f"Error: {e}")
                    sleep(1)
                    if error_patience >= 10:
                        error_patience = 0
                        responses.append(None)
                        break

        # try:
        #     chat_completion = client.chat.completions.create(
        #         messages=prompt,
        #         model=model_name,
        #     )

        #     responses.append(chat_completion.choices[0].message.content)
        # except:
        #     pass


    # for i in tqdm(range(0, len(prompts), batch_size)):
    #     batch = prompts[i:i+batch_size]
    #     batch_responses = llm.invoke(batch)
    #     responses.extend(batch_responses)


    # for response in range(len(responses)):
    #     match = re.search(r"\<\|im_start\|\>assistant", response)
    #     responses[i] = response[match.end():]

    # return responses


In [ ]:
import json

retrieved_df = pd.read_csv(os.path.join(DATA_PATH, "freecad-retrieved-v0/bench_retrieved.csv"))
retrieved_df["retrieved"] = retrieved_df["retrieved"].apply(json.loads)

retrieved = retrieved_df["retrieved"].values
retrieved[0][0]

'Scripting\n\n Arch API and FreeCAD Scripting Basics.\n\nThe Wall tool can be used in macros and from the Python console by using the following function:\n```start_code\nWall = makeWall(baseobj=None, length=None, width=None, height=None, align="Center", face=None, name="Wall")\nend_code```\nCreates a  object from the given , which can be a Draft object, a Sketch, a face, or a solid.\n If no  is given, you can provide the numerical values for the ,  (thickness), and .\n If given,  can be used to give the index of a face from the underlying object, to build this wall on, instead of using the whole object.\n  can be ,  or .\n It returns  if the operation fails.\n\nExample:'

In [ ]:
queries = query_df["X"].values

get_groq_genrated(
    model_name="llama-3.3-70b-versatile",
    queries=queries,
    retrieved=retrieved
)

In [ ]:
with open("generted-v0.json", "w") as f:
    json.dump(responses, f)

In [ ]:
with open("generted-v0.json", "r") as f:
    generated = json.load(f)

# Evaluate generation

In [ ]:
!pip install tree-sitter==0.22.0

  Using cached tree_sitter-0.22.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (10 kB)
Using cached tree_sitter-0.22.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (543 kB)
  Attempting uninstall: tree-sitter
    Found existing installation: tree-sitter 0.24.0
    Uninstalling tree-sitter-0.24.0:
      Successfully uninstalled tree-sitter-0.24.0


In [ ]:
def extract_python_code(generated):
    code_blocks = []
    pattern = re.compile(r"```(?:python)?\n(.*?)```", re.DOTALL | re.IGNORECASE)
    for text in generated:
        if not isinstance(text, str):
            code_blocks.append(None)
            continue
        match = pattern.search(text)
        if match:
            code_blocks.append(match.group(1).strip())
        else:
            code_blocks.append(None)
    return code_blocks

In [ ]:
generated_code = extract_python_code(generated)
generated_code[1]

str

In [ ]:
none_idx = [13, 86, 157, 172]

In [ ]:
query_df.iloc[1]["Y"]

'import Arch\nobj = Arch.makeStructure(length=1000, width=1000, height=3000)'

In [ ]:
from codebleu import calc_codebleu

prediction = "def add ( a , b ) :\n return a + b"
reference = "def sum ( first , second ) :\n return second + first"

result = calc_codebleu([reference], [prediction], lang="python", weights=(0.25, 0.25, 0.25, 0.25), tokenizer=None)
print(result)

{'codebleu': 0.5537842627792564, 'ngram_match_score': 0.10419044640136393, 'weighted_ngram_match_score': 0.11094660471566163, 'syntax_match_score': 1.0, 'dataflow_match_score': 1.0}


In [ ]:
from codebleu import calc_codebleu

result = calc_codebleu(
    [query_df.iloc[1]["Y"]],
    [generated_code[1]],
    lang="python",
    weights=(0.25, 0.25, 0.25, 0.25),
    tokenizer=None
)
print(result)

NameError: name 'query_df' is not defined

# LC

In [ ]:
# class LangchainRAG:
#     def __init__(self, api_key: str, model: str = "llama-3.1-8b-instant"):
#         self.api_key = api_key
#         self.model = model

#         self.qdrant_path = "qdrant"
#         self.collection_name = COLLECTION_NAME

#         self.llm = self._load_llm()
#         self.vectorstore = self._connect_vectorstore()
#         self.qa_chain = self._create_qa_chain()


#     def _load_llm(self):
#         return ChatGroq(
#             api_key=self.api_key,
#             model=self.model,
#             temperature=0.2
#         )

#     def _connect_vectorstore(self):
#         embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": device})
#         vectordb = LCQdrant.from_existing_collection(
#             path="qdrant",
#             embedding=embeddings,
#             collection_name="freecad_docs_lc",
#         )
#         return vectordb

#     def _create_qa_chain(self):
#         retriever = self.vectorstore.as_retriever(search_kwargs={"k": 3})
#         return RetrievalQA.from_chain_type(llm=self.llm, retriever=retriever, chain_type="stuff")

#     def query(self, question: str) -> str:
#         result = self.qa_chain.invoke({"query": question})
#         return result["result"]

   